# RAG — Voice of Customer com Olist

Este notebook executa a jornada completa do projeto: preparação da base, criação de documentos, embeddings, indexação FAISS, recuperação semântica e geração fundamentada por evidências.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from config.settings import settings
from src.data.loader import load_reviews
from src.data.preprocessing import prepare_reviews, build_documents
from src.embeddings.embedder import EmbeddingModel
from src.retrieval.retriever import VectorRetriever
from src.rag.pipeline import RAGPipeline
from src.generation.llm import LLMGenerator

DATA_FILE = settings.data_dir / 'raw' / 'olist_order_reviews_dataset.csv'
INDEX_DIR = settings.data_dir / 'vectorstore'
print(DATA_FILE)


## 1. Exploração e preparação


In [ ]:
df = load_reviews(DATA_FILE)
print('Linhas:', len(df))
print('Colunas:', list(df.columns))
print(df[['review_id','review_score','review_comment_message']].head())
prepared = prepare_reviews(df)
documents = build_documents(prepared)
print('Avaliações com comentário:', len(prepared))
print('Documentos:', len(documents))


## 2. Embeddings e índice vetorial


In [ ]:
embedder = EmbeddingModel(settings.embedding_model)
embeddings = embedder.encode([d['text'] for d in documents])
retriever = VectorRetriever(embeddings, documents)
retriever.save(INDEX_DIR)
print('Dimensão:', embeddings.shape[1])
print('Vetores indexados:', retriever.size)


## 3. Recuperação semântica


In [ ]:
question = 'Quais são os principais problemas relatados pelos clientes sobre a entrega?'
query_embedding = embedder.encode([question])[0]
evidence = retriever.search(query_embedding, top_k=settings.top_k, min_score=settings.min_relevance_score)
for item in evidence:
    print(f"[{item['document_id']}] score={item['score']:.3f} | {item['text']}")


## 4. Geração fundamentada (RAG)

A geração recebe somente as evidências recuperadas e deve citar os IDs utilizados. Sem evidência acima do limiar, o pipeline não fabrica resposta.


In [ ]:
generator = LLMGenerator(settings.llm_model)
pipeline = RAGPipeline(embedder, retriever, generator, settings.top_k, settings.min_relevance_score)
result = pipeline.ask(question)
print(result['answer'])
print('\nEvidências:', [x['document_id'] for x in result['evidence']])


## 5. Consulta fora do escopo / sem evidência


In [ ]:
out_of_scope = 'Qual é a previsão do tempo para amanhã?'
result_out = pipeline.ask(out_of_scope)
print(result_out['answer'])


## 6. Reprodutibilidade

Para reconstruir o índice do zero: `python scripts/build_index.py`. Para consultar via terminal: `python scripts/query_rag.py "sua pergunta"`. Para a interface: `streamlit run app/streamlit_app.py`.
